# Week 10 Problem Set


In [ ]:
%load_ext nb_mypy
%nb_mypy On

In [ ]:
from typing import TypeAlias
from typing import Optional, Any    

Number: TypeAlias = int | float

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.axes as axes
import seaborn as sns
from IPython.display import display

## Cohort Session

**CS0.** *Plot:* Read data for Boston Housing Prices and write a function `get_features_targets()` to get the columns for the features and the targets from the input argument data frame. The function should take in Pandas' dataframe and two lists. The first list is for the feature names and the other list is for the target names. 

We will use the following columns for our test cases:
- x data: RM column - use z normalization (standardization)
- y data: MEDV column

**Make sure you return a new data frame for both the features and the targets.**

We will normalize the feature using z normalization. Plot the data using scatter plot. 



In [ ]:
# STEP 0: normalize the dataset, this is done before plotting or anything else
# clean up the dataset by normalising the features  so that we don't end up with very large parameter values that we need to store
# yhat = b0 + b1x
# we want to make sure our features are between predictable range without eliminating useful information
# because of this we want to normalize them to make them between -1 to 1 
def normalize_z(array: np.ndarray, columns_means: Optional[np.ndarray]=None, 
                columns_stds: Optional[np.ndarray]=None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    assert columns_means is None or columns_means.shape == (1, array.shape[1])
    assert columns_stds is None or columns_stds.shape == (1, array.shape[1])

    if columns_means is None: 
        columns_means: np.ndarray = array.mean(axis=0).reshape(1, -1) # reshape output into 1 by N array shape 
    if columns_stds is None:
        columns_stds: np.ndarray = array.std(axis=0).reshape(1, -1)

    out: np.ndarray = (array - columns_means) / columns_stds
    assert out.shape == array.shape
    assert columns_means.shape == (1, array.shape[1])
    assert columns_stds.shape == (1, array.shape[1])
    return out, columns_means, columns_stds


In [ ]:
def get_features_targets(df: pd.DataFrame, 
                         feature_names: list[str], 
                         target_names: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_feature: pd.DataFrame = df[feature_names] # if feature_names is not list[str] type and just str, then we will get Series and not Dataframe
    df_target: pd.DataFrame = df[target_names]
    return df_feature, df_target

In [ ]:
df: pd.DataFrame = pd.read_csv("housing_processed.csv")
df_feature, df_target = get_features_targets(df,["RM"],["MEDV"])
array_feature,_,_ = normalize_z(df_feature.to_numpy())
# we normalise the features, not the target in this case 
# in the future if we have a new FEATURE (single feature) -- we call this TEST data, e.g new house size, we will need to call normalize_z again with TRAIN SET's mean and std to normalize this new house size, so that they can be PLUGGED into the model and trained parameters: 
# yhat = b0 + b1x, and we get a valid value of predicted target yhat 
# b0 and b1 are trained with normalized feature, so we CANNOT just plug in raw feature value x into the model and expect yhat to be proper

assert isinstance(array_feature, np.ndarray)
assert isinstance(df_target, pd.DataFrame)
assert np.isclose(array_feature.mean(), 0.0)
assert np.isclose(array_feature.std(), 1.0)
assert np.isclose(df_target.mean(), 22.532806)
assert np.isclose(df_target.std(), 9.1971)


In [ ]:
sns.set()
plt.scatter(df_feature, df_target)

**CS1.** *Cost Function:* Write a function `compute_cost_linreg()` to compute the cost function of a linear regression model. The function should take in two 2-D numpy arrays. The first one is the matrix of the linear equation and the second one is the actual target value.

Recall that:

$$J(\hat{\beta}_0, \hat{\beta}_1) = \frac{1}{2m}\Sigma^m_{i=1}\left(\hat{y}(x^i)-y^i\right)^2$$

where

$$\hat{y}(x^i) = \hat{\beta}_0 + \hat{\beta}_1 x^i$$

The function should receive a numpy array, so we will need to convert to numpy array and change the shape. To do this, we will create three other functions:
- `calc_linreg(X, b)`: which is used to calculate the $\hat{y} = Xb$ vector.
- `prepare_feature(df)`: which takes in a two-dimensional numpy array for the feature. The function should also add a column of constant 1s in the first column.

You can use the following methods in your code:
- `df.to_numpy()`: which is to convert a Pandas data frame to Numpy array.
- `np.reshape(row, col)`: which is to reshape the numpy array to a particular shape.
- `np.concatenate((array1, array2), axis)`: which is to join a sequence of arrays along an existing axis.
- `np.matmul(array1, array2)`: which is to do matrix multiplication on two Numpy arrays.
- `np.squeeze()`: to reduce the numpy array to a single number.


In [ ]:
# yhat = b0hat + b1hat * x
def calc_linreg(X: np.ndarray, beta: np.ndarray) -> np.ndarray:
    # X is called the feature matrix 
    # The number of rows is the number of dataset in the trainset 
    # The number of columns is the number of features 
    # yhat = b0 *1 + b1*x 
    # given just 1 dataset,
    # Feature matrix X: [[1 x]] # we are "folding" the b0 multiplier, which is just 1, into the X feature matrix 
    # the beta vector: [b0 b1]
    # [[1 x]] matmul [b0 b1] ---> b0 * 1 + b1 * x
    result = np.matmul(X, beta)
    # we need to make sure that the shape of the result array tallies with X 
    # if we have N data points in the dataset, then the result array should have N x 1 dimension 
    assert result.shape == (X.shape[0], 1)
    return result

$$J(\hat{\beta}_0, \hat{\beta}_1) = \frac{1}{2m}\Sigma^m_{i=1}\left(\hat{y}(x^i)-y^i\right)^2$$


In [ ]:
def compute_cost_linreg(X: np.ndarray, y: np.ndarray, beta: np.ndarray) -> np.ndarray:
   m: int = X.shape[0] # get the number of data in the dataset 
   predicted_y = calc_linreg (X, beta)
   error = predicted_y - y  # this is error is a vector of shape m by 1 
   # eg: error is [1 2 3] --> we want 1^2 + 2^2 + 3^2
   # we can do matmul: [[1 2 3]] ( 1 row 3 columns)  matmul  [1 2 3] (3 rows, 1 col) --> result is 1 row 1 col , e.g: [[14]]
   error_sq = np.matmul(error.T, error)
   J: np.ndarray = (1/(2*m)) * error_sq
   assert J.shape == (1,1) # 1 row 1 column 
   # we want to return scalar, so we need to take out the content of J
   return np.squeeze(J) 


In [ ]:
def prepare_feature(np_feature: np.ndarray) -> np.ndarray:
    # add columns of 1s, which is essentially a "feature" for the b0 parameter and so that we can use np.matmul in our code 
    # get the number of columns
    cols: int = np_feature.shape[1] 
    # create an array of 1s, which shape of m rows and 1 column
    # e.g: np_feature = [[1, 2], [3, 4], [5, 6]] # 3 rows, 2 columns
    # we want to have: [[1, 1, 2], [1, 3, 4], [1, 5, 6]] # append columns of 1s 
    # first we prepare a vector of 1s: [[1], [1], [1]]  # 3 rows, 1 column 
    # then we concatenate the two arrays: [[1], [1], [1]] with [[1, 2], [3, 4], [5, 6]] 
    X:np.ndarray = np.concatenate((np.ones((np_feature.shape[0],1)), np_feature), axis = 1 ) # axis = 1 is to concatenate column wise
    return X

    

In [ ]:
X: np.ndarray = prepare_feature(df_feature.to_numpy())
target: np.ndarray = df_target.to_numpy()

assert isinstance(X, np.ndarray)
assert isinstance(target, np.ndarray)
assert X.shape == (506, 2)
assert target.shape == (506, 1)

In [ ]:
# print(X)
beta: np.ndarray = np.zeros((2,1))
J: np.ndarray = compute_cost_linreg(X, target, beta)
print(J)
assert np.isclose(J, 296.0735)

beta: np.ndarray = np.ones((2,1))
J: np.ndarray = compute_cost_linreg(X, target, beta)
print(J)
assert np.isclose(J, 154.2249)

beta: np.ndarray = np.array([-1, 2]).reshape((2,1))
J: np.ndarray = compute_cost_linreg(X, target, beta)
print(J)
assert np.isclose(J, 94.3256)


**CS2.** *Gradient Descent:* Write a function called `gradient_descent_linreg()` that takes in these parameters:
- `X`: is a 2-D numpy array for the features
- `y`: is a vector array for the target
- `beta`: is a column vector for the initial guess of the parameters
- `alpha`: is the learning rate
- `num_iters`: is the number of iteration to perform

The function should return two numpy arrays:
- `beta`: is coefficient at the end of the iteration
- `J_storage`: is the array that stores the cost value at each iteration

You can use some of the following functions:
- `calc_linreg(X, b)`: which is used to calculate $y = Xb$ vector.
- `np.matmul(array1, array2)`: which is to do matrix multiplication on two Numpy arrays.
- `compute_cost_linreg()`: which the function you created in the previous problem set to compute the cost.

In [ ]:
def gradient_descent_linreg(X: np.ndarray, y: np.ndarray, beta: np.ndarray, alpha: float, num_iters: int) -> tuple[np.ndarray, np.ndarray]:
    # find size of data points
    m: int = X.shape[0]
    # create an array to store error value J at each iteration
    # it is an num_iters of gd x 1 vector
    J_storage: np.ndarray = np.zeros((num_iters,1))
    for n in range(num_iters):
        # compute derivative of error with this current beta
        # don't forget that matmul here "loops" through ALL m datapoints in the train set
        deriv: np.ndarray = np.matmul(X.T, (calc_linreg(X, beta) - y))
        # update the beta to be new beta
        beta = beta - alpha * (1/m) * deriv 
        # compute error value with this new beta
        J_storage[n] = compute_cost_linreg(X, y, beta)

    assert beta.shape == (X.shape[1], 1) # beta is a column vector 
    assert J_storage.shape == (num_iters, 1) 
    return beta, J_storage
        
        
        

In [ ]:
iterations: int = 1500
alpha: float = 0.001
beta: np.ndarray = np.zeros((2,1))

beta, J_storage = gradient_descent_linreg(X, target, beta, alpha, iterations)
print(beta)
assert np.isclose(beta[0], -0.069488, rtol=1e-3)
assert np.isclose(beta[1], 3.6626356, rtol=1e-3)

In [ ]:
J_storage

In [ ]:
plt.plot(J_storage)

**CS3.** *Predict:* Write the function `predict_linreg()` that calculates the straight line equation given the features and its coefficient.
- `predict_linreg()`: this function should standardize the feature using z normalization, change it to a Numpy array, and add a column of constant 1s. You should use `prepare_feature()` for this purpose. Lastly, this function should also call `calc_linreg()` to get the predicted y values.

You can use some of the following functions:
- `calc_linreg(X, beta)`: which is used to calculate the predicted y after X has been normalized and added by a constant.
- `np.matmul(array1, array2)`: which is to do matrix multiplication on two Numpy arrays.
- `normalize_z(df)`: which is to do z normalization on the data frame.

In [ ]:
def predict_linreg(array_feature: np.ndarray, beta: np.ndarray, 
                   means: Optional[np.ndarray]=None, 
                   stds: Optional[np.ndarray]=None) -> np.ndarray:
    assert means is None or means.shape == (1, array_feature.shape[1])
    assert stds is None or stds.shape == (1, array_feature.shape[1])
    norm_data, _, _ = normalize_z(array_feature, means, stds) 
    X: np.ndarray = prepare_feature(norm_data)
    result = calc_linreg(X, beta)
    assert result.shape == (array_feature.shape[0], 1) # assert that the result vector is m by 1, where m is # data points
    return result

In [ ]:
df_feature, buf = get_features_targets(df, ["RM"], [])
beta: np.ndarray = np.array([[22.53279993],[ 6.39529594]]) # from previous result
pred: np.ndarray = predict_linreg(df_feature.to_numpy(), beta)

assert isinstance(pred, np.ndarray)
assert pred.shape == (506, 1)
assert np.isclose(pred.mean(), 22.5328, rtol=1e-3)
print(pred.std())
assert np.isclose(pred.std(), 6.3953, rtol=1e-3)

In [ ]:
plt.plot(df_feature["RM"],target,'o')
plt.plot(df_feature["RM"],pred,'-')

In [ ]:
means: np.ndarray = np.array([6.284634]).reshape(1, -1)
stds: np.ndarray = np.array([0.702617]).reshape(1, -1)
beta: np.ndarray = np.array([[22.53279993],[ 6.39529594]]) # from previous result
input_1row: np.ndarray = np.array([[6.593]])
pred_1row: np.ndarray = predict_linreg(input_1row, beta, means, stds)
assert np.isclose(pred_1row[0][0], 25.33958)

**CS4.** *Splitting data:* Do the following tasks:
- Read RM as the feature and MEDV as the target from the data frame.
- Use Week 9's function `split_data()` to split the data into train and test using `random_state=100` and `test_size=0.3`. 
- Normalize and prepare the features and the target.
- Use the training data set and call `gradient_descent_linreg()` to obtain the `theta`.
- Use the test data set to get the predicted values.

You need to replace the `None` in the code below with other a function call or any other Python expressions. 

In [ ]:
def split_data(df_feature: pd.DataFrame, df_target: pd.DataFrame, 
               random_state: Optional[int]=None, 
               test_size: float=0.5) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    indexes: pd.Index = df_feature.index 
    if random_state != None: # this is just for the tester in Vocareum, we just need to make sure that the randomness remains "predictable" for grading purposes, hence we set the seed 
        np.random.seed(random_state) 
    
    # find the proportion of test set 
    k: int = int(test_size * len(indexes))
    test_index = np.random.choice(indexes, k, replace=False)
    # find the indexes that are not selected by the test index
    train_index = indexes.drop(test_index)
    # time to create the dataframe of feature & target for each set (train & test)
    df_feature_train: pd.DataFrame = df_feature.loc[train_index, :]  
    df_feature_test: pd.DataFrame  = df_feature.loc[test_index, :]
    df_target_train: pd.DataFrame = df_target.loc[train_index, : ]
    df_target_test: pd.DataFrame = df_target.loc[test_index, :]
    # this is not a good practice to return elements like this (positional), we can easily swap between them
    # a better way is to return it in a data structure
    return df_feature_train, df_feature_test, df_target_train, df_target_test
    

In [ ]:
# STEP 0: Dataset preparation
# get features and targets from data frame
df_feature, df_target = get_features_targets(df, ["RM"], ["MEDV"])

# split the data into training and test data sets
df_feature_train, df_feature_test, df_target_train, df_target_test = split_data(df_feature, df_target, random_state = 100, test_size = 0.3)

# normalize the feature using z normalization
array_feature_train_z, means, stds = normalize_z(df_feature_train.to_numpy())

# STEP 2: We know we use Lin Reg
# prepare the X matrix and the target vector as ndarray 
X: np.ndarray = prepare_feature(array_feature_train_z)
target: np.ndarray = df_target_train.to_numpy()

iterations: int = 1500
alpha: float = 0.01
beta: np.ndarray = np.zeros((2,1)) # initial guess of beta, create a vector of 2 rows and 1 column, then fill it up with zeroes

# STEP 3 and 4: find the cost function and perform Grad Descent
# call the gradient_descent function
beta, J_storage = gradient_descent_linreg(X, target, beta, alpha, iterations)

# STEP 6: run on test set, this is supposed to be run exactly ONCE
# call the predict method to get the predicted values
# the means and stds are taken from the TRAIN SET 
# we need to also NORMALIZE the feature test because the beta is trained on normalized features 
# this is all done in predict_linreg 
pred: np.ndarray = predict_linreg(df_feature_test.to_numpy(), beta, means, stds)


print(beta)

In [ ]:

assert isinstance(pred, np.ndarray)
assert pred.shape == (151, 1)
assert np.isclose(pred.mean(), 22.31259, rtol=1e-3)
assert np.isclose(pred.std(), 5.69332, rtol=1e-3)


In [ ]:
plt.scatter(df_feature_test, df_target_test)
plt.plot(df_feature_test, pred, color="orange")

**CS5.** Create a function `build_model_linreg()` that perform the following steps:
- change all data to numpy array.
- normalize the training feature data set using `normalize_z()` function.
- create X matrix.
- run gradient descent by calling `gradient_descent_linreg()` function.

This function should output `model` and `J_storage` where `model` is a dictionary containing `beta`, `means` and `stds`. 

In [ ]:
# just wrap all the code in CS4 in a function 
def build_model_linreg(df_feature_train: pd.DataFrame,
                       df_target_train: pd.DataFrame,
                       beta: Optional[np.ndarray] = None,
                       alpha: float = 0.01,
                       iterations: int = 1500) -> tuple[dict[str, Any], np.ndarray]:
    # check if initial beta values are given
    if beta is None: 
        beta = np.zeros((df_feature_train.shape[1]+1, 1)) # add one dimension to the feature_train array because of the b0 coefficient 
    assert beta.shape == (df_feature_train.shape[1]+1, 1) # to make sure if beta argument is given, then it conforms to the shape of the feature train

    array_feature_train_z, means, stds = normalize_z(df_feature_train.to_numpy())

    # prepare the X matrix and the target vector as ndarray 
    X: np.ndarray = prepare_feature(array_feature_train_z)
    target: np.ndarray = df_target_train.to_numpy()
    beta, J_storage = gradient_descent_linreg(X, target, beta, alpha, iterations)
    # store the output in model dictionary 
    model = {"beta": beta, "means":means, "stds": stds}

    # assert the shapes 
    assert model["beta"].shape == (df_feature_train.shape[1] + 1, 1) # make sure that beta vector is d by 1 
    assert model["means"].shape == (1, df_feature_train.shape[1]) # make sure that the means vector is also d-1 by 1 (1 per feature)
    assert model["stds"].shape == (1, df_feature_train.shape[1])  # make sure that the stds vector is also d-1 by 1 (1 per feature)
    assert J_storage.shape == (iterations, 1) # make sure we have recorded #iterations of error
    return model, J_storage 


In [ ]:
df_feature, df_target = get_features_targets(df, ["RM"], ["MEDV"])
df_feature_train, df_feature_test, df_target_train, df_target_test = split_data(df_feature, df_target, random_state=100, test_size=0.3)
model, J_storage = build_model_linreg(df_feature_train, df_target_train)
model
assert isinstance(model, dict)
assert "beta" in model
assert "means" in model
assert "stds" in model
assert model['beta'].shape == (2, 1)
assert np.isclose(model['beta'][0, 0], 22.66816258)  
assert np.isclose(model['beta'][1, 0], 6.26923893) 
assert np.isclose(model['means'], 6.2968338)
assert np.isclose(model['stds'], 0.72077827)

In [ ]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**CS6.** *R2 Coefficient of Determination:* Write a function to calculate the coefficient of determination as given by the following equations.

$$r^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

where

$$SS_{res} = \Sigma_{i=1}^n (y_i - \hat{y}_i)^2$$ 

where $y_i$ is the actual target value and $\hat{y}_i$ is the predicted target value.

$$SS_{tot} = \Sigma_{i=1}^n (y_i - \overline{y})^2$$

where 
$$ \overline{y} = \frac{1}{n} \Sigma_{i=1}^n y_i$$
and $n$ is the number of target values.

You can use the following functions in your code:
- `np.mean(array)`: which is to get the mean of the array. You can also call it using `array.mean()`.
- `np.sum(array)`: which is to sum the array along a default axis. You can specify which axis to perform the summation.

In [ ]:
def r2_score(y: np.ndarray, ypred: np.ndarray) -> float:
    ymean: np.ndarray = np.mean(y)
    diff: np.ndarray = y - ymean 
    sstot: np.ndarray = np.matmul(diff.T, diff)
    error: np.ndarray = y - ypred 
    ssres: np.ndarray = np.matmul(error.T, error)
    return 1 - np.squeeze(ssres/sstot) # remember to squeeze the value out of the matrix form because r^2 is a scalar, not a 1-element vector [[r^2]] 
    

In [ ]:
target: np.ndarray = df_target_test.to_numpy()
r2: float = r2_score(target, pred)
print(r2)
assert np.isclose(r2, 0.447557, rtol=1e-3)

In [ ]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**CS7.** *Mean Squared Error:* Create a function to calculate the MSE as given below.

$$MSE = \frac{1}{n}\Sigma_{i=1}^n(y^i - \hat{y}^i)^2$$


In [ ]:
def mean_squared_error(target: np.ndarray, pred: np.ndarray) -> float:
    m: int = target.shape[0] # number of data points 
    error = target-pred 
    return 1/m * np.squeeze(np.matmul(error.T, error))

In [ ]:
mse: float = mean_squared_error(target, pred)
print(mse)
assert np.isclose(mse, 54.2684, rtol=1e-3)

In [ ]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**CS8.** *Optional:* Redo the above tasks using Sci-kit learn libraries. You will need to use the following:
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score as sk_r2_score
from sklearn.metrics import mean_squared_error as sk_mse

In [ ]:
# Read the CSV and extract the features
# df: pd.DataFrame = None
# df_feature, df_target = None, None
# normalize
# df_feature, _, _ = None, None, None

### BEGIN SOLUTION
df: pd.DataFrame = pd.read_csv("housing_processed.csv")
df_feature, df_target = get_features_targets(df, ["RM"], ["MEDV"])
df_feature,_,_ = normalize_z(df_feature.to_numpy())
### END SOLUTION

In [ ]:
# Split the data into training and test data set using scikit-learn function
# df_feature_train, df_feature_test, df_target_train, df_target_test = None, None, None, None

# Instantiate LinearRegression() object
# model: LinearRegression = None

# Call the fit() method
# pass

### BEGIN SOLUTION
df_feature_train, df_feature_test, df_target_train, df_target_test = train_test_split(df_feature, df_target, random_state=100, test_size=0.3)

sk_model: LinearRegression = LinearRegression()
sk_model.fit(df_feature_train, df_target_train)
### END SOLUTION

print(sk_model.coef_, sk_model.intercept_)
assert np.isclose(sk_model.coef_,[[6.04492]])
assert np.isclose(sk_model.intercept_, 22.52999668)

In [ ]:
# Call the predict() method
# pred: np.ndarray = None

### BEGIN SOLUTION
pred: np.ndarray = sk_model.predict(df_feature_test)
### END SOLUTION

print(type(pred), pred.mean(), pred.std())
assert isinstance(pred, np.ndarray)
assert np.isclose(pred.mean(), 22.361699)
assert np.isclose(pred.std(), 5.7011267)

In [ ]:
plt.scatter(df_feature_test, df_target_test)
plt.plot(df_feature_test, pred, color="orange")

In [ ]:
r2: float = sk_r2_score(df_target_test, pred)
print(r2)
assert np.isclose(r2, 0.457647)

In [ ]:
mse: float = sk_mse(df_target_test, pred)
print(mse)
assert np.isclose(mse, 54.93216)